# Part 1. Equation of a Slime

How many late days are you using for this assignment?

3

In [179]:
# Imports section
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn import svm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.neural_network import MLPClassifier

## 1. Loading the dataset

In [180]:
# Using pandas load the dataset
science_data = pd.read_csv("science_data_large.csv")
# Output the first 15 rows of the data
science_data.head(15)
# Display a summary of the table information (number of datapoints, etc.)
science_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


## 2. Splitting the dataset

In [181]:
# Take the pandas dataset and split it into our features (X) and label (y)
# features are temperature and mols kcl, and y is size
X, y = science_data[["Temperature °C", "Mols KCL"]], science_data["Size nm^3"]
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
# For grading consistency use random_state=42 

## 3. Perform a Linear Regression

In [182]:
# Use sklearn to train a model on the training set
reg = LinearRegression().fit(X_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample_data = pd.DataFrame([[300, 500]], columns=["Temperature °C", "Mols KCL"])
prediction = reg.predict(sample_data)
print("Predicted Size nm^3:", prediction)
# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = reg.score(X_test, y_test)
print("Score:", score)
# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = reg.coef_
intercept = reg.intercept_

# Display the coefficients and intercept
print("Coefficients:", coefficients)
print("Intercept:", intercept)


Predicted Size nm^3: [366799.97767107]
Score: 0.8552472077276096
Coefficients: [ 866.14641337 1032.69506649]
Intercept: -409391.4795834081


Write the linear equation of a slime: (example equation: $E = mc^2$)

$$
h(x) = w_0 + w_1 X_{\text{Temperature}} + w_2 X_{\text{Mols}}
$$
where h(x) is the Predicted Size nm^3.

With the above sample data point:

$$
366799.97767107 = w_0 + w_1 * 300 + w_2 * 500
$$

Report on score and explain meaning:

The score is 0.86, rounded up to 2dp. It means that the regression explains 86% of the variance in the dataset. 

## 4. Use Cross Validation

In [183]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42
# Define the K-Fold cross-validator
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Perform cross-validation
cv_scores = cross_val_score(reg, X, y, cv=kf, scoring='r2')

# Report on their finding and their significance
print("CV Score", cv_scores)
# Calculate mean and standard deviation
mean_score = cv_scores.mean()
std_dev = cv_scores.std()

print("mean CV Score",mean_score)
print("std of CV Score",std_dev)


CV Score [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
mean CV Score 0.8597294202684646
std of CV Score 0.018387737139306408


Write findings here:

The R² scores indicate the proportion of variance in the dependent variable that is predictable from the independent variables. A score of 1.0 indicates perfect prediction, while a score of 0.0 indicates that the model does not explain any variance.

In this case, the mean R² score of approximately \(0.8534\) suggests that the model explains around 85.34% of the variance in the target variable. This is indicative of a good fit, confirming that the model performs well across different subsets of the data.

The standard deviation of the scores is about \(0.0184\), which indicates that the model's performance is relatively stable across the different folds. A low standard deviation implies that our results are consistent, and we can be confident in the model's predictive capabilities.

## 5. Using Polynomial Regression

In [184]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly_model = Pipeline([('poly', PolynomialFeatures(degree=2)),
                  ('linear', LinearRegression())])
poly_model = poly_model.fit(X_train, y_train)

# Perform k-fold cross validation (as above)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_poly = cross_val_score(poly_model, X, y, cv=kf, scoring='r2')
# Report on the metrics and output the resultant equation as you did in Part 3.
print("CV Score", cv_scores_poly)
# Calculate mean and standard deviation
mean_score = cv_scores_poly.mean()
std_dev = cv_scores_poly.std()

print("mean CV Score",mean_score)
print("std of CV Score",std_dev)

CV Score [1. 1. 1. 1. 1.]
mean CV Score 1.0
std of CV Score 0.0


In [185]:
# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
linear_model = poly_model.named_steps['linear']  # Access the LinearRegression part of the pipeline
coefficients_poly = linear_model.coef_  # Extract the coefficients
intercept_poly = linear_model.intercept_  # Extract the intercept

# Display the coefficients and intercept
print("Coefficients:", coefficients_poly)
print("Intercept:", intercept_poly)

Coefficients: [ 0.00000000e+00  1.20000000e+01 -1.27198206e-07  1.26473309e-11
  2.00000000e+00  2.85714287e-02]
Intercept: 2.047908492386341e-05


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$$
y = 2.0479 + 12.0 \cdot X_{\text{temp}} + 2.0 \cdot X_{\text{Mols}} + 0.0285714287 \cdot X_{\text{temp}} \cdot X_{\text{Mols}}
$$

Report on the score and interpret:

The R² scores indicate the proportion of variance in the dependent variable that is predictable from the independent variables. A score of \(1.0\) signifies perfect prediction, meaning that the model explains all the variance in the target variable.

In this case, the mean R² score of \(1.0\) suggests that our model perfectly predicts the target variable across all folds of the cross-validation. 

Additionally, the standard deviation of \(0.0\) indicates that there is no variation in the model’s performance across different subsets of the data. This suggests that the model is perfectly fitting the training data, yielding identical predictions for each fold.



# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

## a) Loading the dataset

In [186]:
# Load the dataset. Then train and evaluate the classification models.

# Using pandas load the dataset
ckd_data = pd.read_csv("ckd_feature_subset.csv")
# Output the first 15 rows of the data
ckd_data.head(15)
# Display a summary of the table information (number of datapoints, etc.)
ckd_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         153 non-null    float64
 1   bp          153 non-null    float64
 2   wbcc        153 non-null    float64
 3   appet_poor  153 non-null    int64  
 4   appet_good  153 non-null    int64  
 5   rbcc        153 non-null    float64
 6   Target_ckd  153 non-null    int64  
dtypes: float64(4), int64(3)
memory usage: 8.5 KB


## b) split the dataset

In [187]:
#split data set
# Take the pandas dataset and split it into our features (X) and label (y)
X, y = ckd_data[["age", "bp", "wbcc", "appet_poor", "appet_good", "rbcc"]], ckd_data["Target_ckd"]
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)


## c) Logistic Regression

In [188]:
#logistic regression
log_reg = LogisticRegression(random_state=42).fit(X_train, y_train)
#get the score
score = log_reg.score(X_test, y_test)
print("Score:", score)


Score: 0.6875


In [189]:
#cross valudation for Logistic Regression
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_log = cross_val_score(log_reg, X, y, cv=kf, scoring='r2')
cv_scores_log



array([ 0.11428571,  0.68686869, -0.03333333,  0.06832298,  0.58333333])

## d) Support Vector Machine

In [190]:
#support vector machines
svm_model = svm.SVC()
svm_model.fit(X_train, y_train)
score = log_reg.score(X_test, y_test)
print("Score:", score)


Score: 0.6875


In [191]:
#cross validation for support vector machine
cv_scores_svm = cross_val_score(svm_model, X, y, cv=kf, scoring='r2')
cv_scores_svm

array([0.55714286, 1.        , 0.40952381, 0.44099379, 0.79166667])

## e) K nearest neighbors

In [192]:
#k nearest neighbors
knn = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=11))
])

knn.fit(X_train, y_train)
accuracy = knn.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.88


In [193]:
#cross validation for k nearest neighbors
cv_scores_knn = cross_val_score(knn, X, y, cv=kf, scoring='r2')
cv_scores_knn

array([ 0.55714286,  0.84343434, -0.03333333,  0.25465839,  0.79166667])

## f) Neural Networks

In [194]:
#neural networks 
nn = MLPClassifier(solver='lbfgs', alpha=1e-5,
                    hidden_layer_sizes=(5, 2), random_state=42)
nn.fit(X_train, y_train)
accuracy = nn.score(X_test, y_test)
print(f"Model accuracy: {accuracy:.2f}")

Model accuracy: 0.88


In [195]:
#cv score
cv_scores_nn = cross_val_score(nn, X, y, cv=kf, scoring='r2')
cv_scores_nn

array([0.55714286, 0.68686869, 0.40952381, 0.8136646 , 0.79166667])

## g) Score Reports

In [196]:

results = {
    "Model": [
        "Logistic Regression",
        "Support Vector Machines",
        "k-Nearest Neighbors",
        "Neural Networks",
    ],
    "Mean Score": [
        round(pd.Series(cv_scores_log).mean(), 4),  # rounding for better readability
        round(pd.Series(cv_scores_svm).mean(), 4),
        round(pd.Series(cv_scores_knn).mean(), 4),
        round(pd.Series(cv_scores_nn).mean(), 4),
    ],
    "Standard Deviation": [
        round(pd.Series(cv_scores_log).std(), 4),
        round(pd.Series(cv_scores_svm).std(), 4),
        round(pd.Series(cv_scores_knn).std(), 4),
        round(pd.Series(cv_scores_nn).std(), 4),
    ]
}


scores_df = pd.DataFrame(results)


print(scores_df)

                     Model  Mean Score  Standard Deviation
0      Logistic Regression      0.2839              0.3271
1  Support Vector Machines      0.6399              0.2511
2      k-Nearest Neighbors      0.4827              0.3706
3          Neural Networks      0.6518              0.1693


### report: 

1. **Neural Networks**:
   - **Mean Score**: \(0.6518\)
   - **Standard Deviation**: \(0.1693\)
   - The Neural Networks model achieved the highest mean score, it was the most effective in predicting the target variable.
   - The standard deviation is relatively low, indicating consistent performance across the folds.

2. **Support Vector Machines (SVM)**: 
   - **Mean Score**: \(0.6399\)
   - **Standard Deviation**: \(0.2511\)
   - SVM also performed well but had a slightly lower mean score compared to Neural Networks. 
   - The higher standard deviation was worser than the neural networks.

3. **k-Nearest Neighbors (KNN)**: 
   - **Mean Score**: \(0.4827\)
   - **Standard Deviation**: \(0.3706\)
   - KNN showed moderate performance. 
   - The standard deviation indicates that its predictions were less stable compared to the Neural Networks and SVM models.

4. **Logistic Regression**: 
   - **Mean Score**: \(0.2839\)
   - **Standard Deviation**: \(0.3271\)
   - Logistic Regression had the lowest mean score, reflecting that it struggled to perform well in this task.
   - The relatively high standard deviation suggests a lack of consistency in its predictions.

In summary, the Neural Networks model outperformed the other models in terms of mean score. I think that it was able to perform better because it was able to extract more features of the relationship between the input and the output through its hidden layers. The logistic regression that had worked well with the slime, did not show the same strength when applied to the ckd model. 

## Results and Conclusion for Classification Experiments

### neural networks settings 1

In [197]:
#neural network settings 1
nn = MLPClassifier(solver='lbfgs', alpha=1e-5,
                    hidden_layer_sizes=(5, 2), random_state=42)
nn.fit(X_train, y_train)

#cv score
cv_scores_nn = cross_val_score(nn, X, y, cv=kf, scoring='r2')
cv_scores_nn

array([0.55714286, 0.68686869, 0.40952381, 0.8136646 , 0.79166667])

### neural networks settings 2

In [198]:
#neural networks setting 2, changing the solver to sgd --> stochastic gradient descent
nn2 = MLPClassifier(solver='sgd', alpha=1e-5,
                    hidden_layer_sizes=(5, 2), random_state=42)
nn2.fit(X_train, y_train)

#cv score
cv_scores_nn2 = cross_val_score(nn2, X, y, cv=kf, scoring='r2')
cv_scores_nn2


array([-0.47619048, -0.40909091, -0.47619048, -0.30434783, -0.25      ])

### neural network settings 3

In [199]:
#neural networks setting 3, changing the hiddin layer
nn3 = MLPClassifier(solver='lbfgs', alpha=1e-5,
                    hidden_layer_sizes=(10, 2), random_state=42)
nn3.fit(X_train, y_train)

#cv score
cv_scores_nn3 = cross_val_score(nn3, X, y, cv=kf, scoring='r2')
cv_scores_nn3


array([0.85238095, 0.68686869, 0.7047619 , 1.        , 0.79166667])

In [200]:
# Experiments with Neural Network.
experiments = {
    "Model": [
        "Neural Networks",
        "Neural Networks, Solver: SGD ",
        "Neural Networks, Hidden Layer: 10,2",
    ],
    "Mean Score": [
        round(pd.Series(cv_scores_nn).mean(), 4),  # rounding for better readability
        round(pd.Series(cv_scores_nn2).mean(), 4),
        round(pd.Series(cv_scores_nn3).mean(), 4),
    ],
    "Standard Deviation": [
        round(pd.Series(cv_scores_nn).std(), 4),
        round(pd.Series(cv_scores_nn2).std(), 4),
        round(pd.Series(cv_scores_nn3).std(), 4),

    ]
}


experiment_scores = pd.DataFrame(experiments)


print(experiment_scores)

                                 Model  Mean Score  Standard Deviation
0                      Neural Networks      0.6518              0.1693
1        Neural Networks, Solver: SGD      -0.3832              0.1024
2  Neural Networks, Hidden Layer: 10,2      0.8071              0.1269


## Results and Conclusion for Neural Network Experiments

1. **Neural Networks (Base Configuration)**:
   - **Mean Score**: \(0.6518\)
   - **Standard Deviation**: \(0.1693\)
   - This base configuration achieved a moderate mean score, demonstrating reasonable predictive performance. 
   - The standard deviation indicates some variability in performance across different folds.

2. **Neural Networks, Solver: SGD (Stochastic Gradient Descent)**:
   - **Mean Score**: \(-0.3832\)
   - **Standard Deviation**: \(0.1024\)
   - The use of Stochastic Gradient Descent (SGD) as a solver resulted in a negative mean score, which means that the models results were way off from the true results.

3. **Neural Networks, Hidden Layer: 10,2**:
   - **Mean Score**: \(0.8071\)
   - **Standard Deviation**: \(0.1269\)
   - This configuration, which incorporated two hidden layers (one with 10 neurons and another with 2 neurons), performed the best among the tested configurations. 
   - The mean score of \(0.8071\) indicates strong predictive capability, while the standard deviation shows that performance was relatively consistent across folds.


The biggest change in results was changing the solver. It made the results worse off. On the other hand, increasing the number of neurons in the first layer worked better. 
